# Pi 5 analysis — visualization-ready

Visualization-ready analysis export. This notebook normalizes all benchmark sampler outputs into consistent metric names and exports tables for plotting.

In [1]:

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 240)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

DATASETS = ["mmlu", "hellaswag", "gsm8k", "dolly"]
SCORED_DATASETS = ["mmlu", "hellaswag", "gsm8k"]

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
print("ROOT:", ROOT)

PLATFORM = "pi5"
RESULTS_DIR = ROOT / "results" / "pi5"
PROCESSED_DIR = RESULTS_DIR / "processed" / "pi5"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("RESULTS_DIR:", RESULTS_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)


ROOT: /home/stephane/llm-experiments
RESULTS_DIR: /home/stephane/llm-experiments/results/pi5
PROCESSED_DIR: /home/stephane/llm-experiments/results/pi5/processed/pi5


## Discover raw benchmark runs

In [2]:

def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()
    except Exception as exc:
        warnings.warn(f"Could not read {path}: {exc}")
        return pd.DataFrame()


def looks_like_run_dir(path: Path) -> bool:
    return path.is_dir() and any((path / f"{d}.csv").exists() or (path / f"{d}_samples.csv").exists() for d in DATASETS)


def discover_runs(results_dir: Path) -> pd.DataFrame:
    rows = []
    if not results_dir.exists():
        return pd.DataFrame(columns=["platform", "model_dir", "run_id", "run_dir"])
    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir() or model_dir.name == "processed":
            continue
        for run_dir in sorted(model_dir.iterdir()):
            if looks_like_run_dir(run_dir):
                rows.append({
                    "platform": PLATFORM,
                    "model_dir": model_dir.name,
                    "run_id": run_dir.name,
                    "run_dir": str(run_dir),
                })
    return pd.DataFrame(rows)


def load_raw_tables(runs_df: pd.DataFrame):
    summary_parts, sample_parts = [], []
    for _, run in runs_df.iterrows():
        run_dir = Path(run["run_dir"])
        for dataset in DATASETS:
            summary_path = run_dir / f"{dataset}.csv"
            samples_path = run_dir / f"{dataset}_samples.csv"
            sdf = read_csv_if_exists(summary_path)
            if not sdf.empty:
                sdf = sdf.copy()
                sdf["platform"] = PLATFORM
                sdf["dataset"] = dataset
                sdf["run_id"] = run["run_id"]
                sdf["run_dir"] = str(run_dir)
                sdf["model_dir"] = run["model_dir"]
                sdf["source_file"] = str(summary_path)
                summary_parts.append(sdf)
            sam = read_csv_if_exists(samples_path)
            if not sam.empty:
                sam = sam.copy()
                sam["platform"] = PLATFORM
                sam["dataset"] = dataset
                sam["run_id"] = run["run_id"]
                sam["run_dir"] = str(run_dir)
                sam["model_dir"] = run["model_dir"]
                sam["source_file"] = str(samples_path)
                sample_parts.append(sam)
    return (
        pd.concat(summary_parts, ignore_index=True) if summary_parts else pd.DataFrame(),
        pd.concat(sample_parts, ignore_index=True) if sample_parts else pd.DataFrame(),
    )

runs_df = discover_runs(RESULTS_DIR)
print("Discovered runs:", len(runs_df))
display(runs_df.head(30))

Discovered runs: 18


,platform,model_dir,run_id,run_dir
0,pi5,Phi-4-mini-instruct-Q4_K_M,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...
1,pi5,Phi-4-mini-instruct-Q4_K_M,2026-05-01_09-41-37,/home/stephane/llm-experiments/results/pi5/Phi...
2,pi5,Phi-4-mini-instruct-Q4_K_M,2026-05-01_15-01-51,/home/stephane/llm-experiments/results/pi5/Phi...
3,pi5,Phi-4-mini-instruct-Q4_K_M,2026-06-27_18-19-44,/home/stephane/llm-experiments/results/pi5/Phi...
4,pi5,Phi-4-mini-instruct-Q4_K_M,2026-06-29_16-02-22,/home/stephane/llm-experiments/results/pi5/Phi...
5,pi5,SmolLM3-3B-Q4_K_M,2026-04-29_13-59-52,/home/stephane/llm-experiments/results/pi5/Smo...
6,pi5,SmolLM3-3B-Q4_K_M,2026-05-01_11-34-49,/home/stephane/llm-experiments/results/pi5/Smo...
7,pi5,SmolLM3-3B-Q4_K_M,2026-05-01_20-21-10,/home/stephane/llm-experiments/results/pi5/Smo...
8,pi5,SmolLM3-3B-Q4_K_M,2026-06-27_18-19-44,/home/stephane/llm-experiments/results/pi5/Smo...
9,pi5,SmolLM3-3B-Q4_K_M,2026-06-29_16-02-22,/home/stephane/llm-experiments/results/pi5/Smo...


## Load raw summary and sample CSVs

In [3]:
raw_summary, raw_samples = load_raw_tables(runs_df)
print("raw_summary", raw_summary.shape)
print("raw_samples", raw_samples.shape)
display(raw_summary.head())
display(raw_samples.head())

raw_summary (24, 50)
raw_samples (17800, 32)


,Model name,Model path,Served model id,All served model ids,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak,platform,dataset,run_id,run_dir,model_dir,source_file,Dolly Served model id,Dolly All served model ids,Dolly TTFT ms mean,Dolly TTFT ms sd,Dolly End-to-end latency ms mean,Dolly End-to-end latency ms sd,Dolly Tokens/sec mean,Dolly Tokens/sec sd,Dolly CPU percent avg mean,Dolly CPU percent avg sd,Dolly Temp C avg mean,Dolly Temp C avg sd,Dolly Sequential sustained tokens/sec mean,Dolly Sequential sustained tokens/sec sd,Dolly Sequential sustained samples/sec mean,Dolly Sequential sustained samples/sec sd,Dolly CPU percent peak,Dolly RAM used MB peak,Dolly Process RAM MB peak,Dolly Temp C peak
0,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf | Phi-4-mini-i...,0.564,0.0117,5798.4835,23.7815,6552.3542,31.5635,3.5501,0.0369,72.3463,0.1756,67.0221,0.5420,0.3417,0.0037,0.1325,0.0006,100.0,6517.08,7265.75,71.9,pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf | Phi-4-mini-i...,0.489,0.0088,8085.0183,26.8886,8861.6988,28.6326,3.6275,0.0305,72.6253,0.3691,67.2594,0.1135,0.2997,0.0034,0.1015,0.0003,100.0,6939.64,7507.83,71.4,pi5,hellaswag,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pi5,dolly,2026-05-01_09-41-37,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,4638.8217,0.0000,37395.0564,0.0000,3.2370,0.0000,97.1212,0.0000,66.5617,0.000,2.7941,0.0000,0.0254,0.0000,100.0,6771.20,7311.20,71.4
3,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,0.640,0.0000,2879.2924,0.0000,42777.9699,0.0000,3.0606,0.0000,98.5461,0.0000,66.6654,0.0000,2.7010,0.0000,0.0223,0.0000,100.0,7010.58,7236.72,69.7,pi5,gsm8k,2026-05-01_15-01-51,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pi5,dolly,2026-06-27_18-19-44,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf | Phi-4-mini-i...,5242.0683,813.6546,39678.4673,2746.5449,3.3266,0.0385,97.0939,0.2332,69.1893,0.432,2.7969,0.0806,0.0241,0.0015,100.0,7674.98,7565.58,75.2


,model_name,served_model_id,model_path,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,system_metrics_trace_json,platform,dataset,run_id,run_dir,model_dir,source_file
0,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,mmlu,1,1,A disk of radius 0.1 m initially at rest under...,A disk of radius 0.1 m initially at rest under...,0.4 m || 0.8 m || 1.2 m || 1.6 m,NaN,NaN,3.0,C,False,C. 1.,2,4587.0487,5453.5665,2.3081,78.2286,100.0,5054.59,6070.88,62.1000,63.7,"[{""cpu_percent"":0.7,""ram_used_mb"":5054.59,""ram...",pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...
1,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,mmlu,1,2,"Cultural homogenization.\n\nA. 1,3,4\nB. 1,2,3...",Cultural homogenization.,"1,3,4 || 1,2,3 || 1,2,3,4 || 2,3,4",NaN,NaN,1.0,B,True,"B. 1,",2,2882.0211,3741.2235,2.3277,71.0143,100.0,5066.02,6078.38,62.3571,63.7,"[{""cpu_percent"":1.7,""ram_used_mb"":5049.56,""ram...",pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...
2,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,mmlu,1,3,Monte Alban is located in the ________ and was...,Monte Alban is located in the ________ and was...,Valley of Mexico; Aztec. || Valley of Oaxaca; ...,NaN,NaN,1.0,A,False,A. Valley of Mexico,4,3405.3366,4271.1300,4.6200,71.0429,100.0,5105.27,6098.95,62.3429,63.7,"[{""cpu_percent"":0.5,""ram_used_mb"":5060.81,""ram...",pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...
3,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,mmlu,1,4,During a clinical study examining the effects ...,During a clinical study examining the effects ...,Pulse pressure || Stroke volume || Systolic bl...,NaN,NaN,3.0,C,False,C. Systolic,2,5416.0967,6283.6773,2.3053,70.7857,100.0,5095.39,6107.67,63.0429,64.8,"[{""cpu_percent"":0.0,""ram_used_mb"":5095.39,""ram...",pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...
4,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,mmlu,1,5,A 17-year-old girl is brought to the physician...,A 17-year-old girl is brought to the physician...,Hyperthyroidism || Hypogonadotropic hypogonadi...,NaN,NaN,1.0,C,False,C. Hypothyroidism,2,8947.6761,9823.7464,2.2829,70.9857,100.0,5102.88,6107.67,63.6143,65.9,"[{""cpu_percent"":0.2,""ram_used_mb"":5092.81,""ram...",pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...


## Normalize schema and derive visualization metrics

In [4]:

def collapse_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Merge duplicate column names by taking the first non-null value row-wise."""
    if df.empty:
        return df.copy()
    out = pd.DataFrame(index=df.index)
    for col in dict.fromkeys(df.columns):
        block = df.loc[:, df.columns == col]
        if block.shape[1] == 1:
            out[col] = block.iloc[:, 0]
        else:
            out[col] = block.bfill(axis=1).iloc[:, 0]
    return out.reset_index(drop=True)


def first_existing(df: pd.DataFrame, candidates, default=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return pd.Series(default, index=df.index)


def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def to_bool_or_nan(s):
    if s is None:
        return np.nan
    if not isinstance(s, pd.Series):
        return s
    def conv(x):
        if pd.isna(x):
            return np.nan
        if isinstance(x, bool):
            return x
        text = str(x).strip().lower()
        if text in {"true", "1", "yes", "y"}:
            return True
        if text in {"false", "0", "no", "n"}:
            return False
        return np.nan
    return s.map(conv)


def safe_div(num, den):
    num = to_num(num)
    den = to_num(den)
    return np.where((den.notna()) & (den != 0), num / den, np.nan)


def add_common_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = collapse_duplicate_columns(df)
    if df.empty:
        return df
    if "platform" not in df.columns:
        df["platform"] = PLATFORM
    if "dataset" in df.columns:
        df["dataset"] = df["dataset"].astype(str).str.lower()
    if "model" not in df.columns:
        df["model"] = first_existing(df, ["model_name", "Model name", "model_alias", "Model alias", "model_dir", "model_alias_dir", "Repo id", "repo_id"])
    df["model"] = df["model"].astype(str)
    df["model_label"] = df["platform"].astype(str) + " | " + df["model"].astype(str)
    if "run_id" in df.columns:
        df["run_label"] = df["platform"].astype(str) + " | " + df["model"].astype(str) + " | " + df["run_id"].astype(str)
    return df

def normalize_summary(raw: pd.DataFrame) -> pd.DataFrame:
    raw = add_common_labels(raw)
    if raw.empty:
        return raw
    out = pd.DataFrame(index=raw.index)
    for col in ["platform", "dataset", "run_id", "run_dir", "model", "model_label", "model_dir", "source_file"]:
        if col in raw.columns:
            out[col] = raw[col]

    # llama.cpp summaries use human-readable names; Dolly may be prefixed with "Dolly ".
    out["accuracy"] = to_num(first_existing(raw, ["Accuracy mean", "Dolly Accuracy mean"]))
    out["accuracy_sd"] = to_num(first_existing(raw, ["Accuracy sd", "Dolly Accuracy sd"]))
    out["ttft_ms"] = to_num(first_existing(raw, ["TTFT ms mean", "Dolly TTFT ms mean"]))
    out["ttft_ms_sd"] = to_num(first_existing(raw, ["TTFT ms sd", "Dolly TTFT ms sd"]))
    out["latency_ms"] = to_num(first_existing(raw, ["End-to-end latency ms mean", "Dolly End-to-end latency ms mean"]))
    out["latency_ms_sd"] = to_num(first_existing(raw, ["End-to-end latency ms sd", "Dolly End-to-end latency ms sd"]))
    out["tokens_per_sec"] = to_num(first_existing(raw, ["Tokens/sec mean", "Dolly Tokens/sec mean"]))
    out["tokens_per_sec_sd"] = to_num(first_existing(raw, ["Tokens/sec sd", "Dolly Tokens/sec sd"]))
    out["sustained_tokens_per_sec"] = to_num(first_existing(raw, ["Sequential sustained tokens/sec mean", "Dolly Sequential sustained tokens/sec mean"]))
    out["sustained_tokens_per_sec_sd"] = to_num(first_existing(raw, ["Sequential sustained tokens/sec sd", "Dolly Sequential sustained tokens/sec sd"]))
    out["samples_per_sec"] = to_num(first_existing(raw, ["Sequential sustained samples/sec mean", "Dolly Sequential sustained samples/sec mean"]))
    out["samples_per_sec_sd"] = to_num(first_existing(raw, ["Sequential sustained samples/sec sd", "Dolly Sequential sustained samples/sec sd"]))
    out["cpu_percent_avg"] = to_num(first_existing(raw, ["CPU percent avg mean", "Dolly CPU percent avg mean"]))
    out["cpu_percent_avg_sd"] = to_num(first_existing(raw, ["CPU percent avg sd", "Dolly CPU percent avg sd"]))
    out["cpu_percent_peak"] = to_num(first_existing(raw, ["CPU percent peak", "Dolly CPU percent peak"]))
    out["ram_used_mb_peak"] = to_num(first_existing(raw, ["RAM used MB peak", "Dolly RAM used MB peak"]))
    out["process_ram_mb_peak"] = to_num(first_existing(raw, ["Process RAM MB peak", "Dolly Process RAM MB peak"]))
    out["temperature_c_avg"] = to_num(first_existing(raw, ["Temp C avg mean", "Dolly Temp C avg mean"]))
    out["temperature_c_avg_sd"] = to_num(first_existing(raw, ["Temp C avg sd", "Dolly Temp C avg sd"]))
    out["temperature_c_peak"] = to_num(first_existing(raw, ["Temp C peak", "Dolly Temp C peak"]))

    out["has_accuracy"] = out["dataset"].isin(SCORED_DATASETS) & out["accuracy"].notna()
    out["accuracy_per_second"] = safe_div(out["accuracy"], out["latency_ms"] / 1000.0)
    out["accuracy_per_1k_ms"] = safe_div(out["accuracy"], out["latency_ms"] / 1000.0)  # same unit as per-second utility
    return out


def normalize_samples(raw: pd.DataFrame) -> pd.DataFrame:
    raw = add_common_labels(raw)
    if raw.empty:
        return raw
    out = pd.DataFrame(index=raw.index)
    keep_cols = [
        "platform", "dataset", "run_id", "run_dir", "model", "model_label", "model_dir", "source_file",
        "served_model_id", "model_path", "sample_idx", "prompt", "question", "choices", "instruction", "context", "gold_answer",
        "parsed_prediction", "raw_output", "system_metrics_trace_json",
    ]
    for col in keep_cols:
        if col in raw.columns:
            out[col] = raw[col]
    out["correct"] = to_bool_or_nan(first_existing(raw, ["correct"]))
    out["accuracy"] = out["correct"].astype(float)
    out.loc[~out["dataset"].isin(SCORED_DATASETS), "accuracy"] = np.nan
    out["output_tokens"] = to_num(first_existing(raw, ["output_tokens"]))
    out["ttft_ms"] = to_num(first_existing(raw, ["ttft_ms"]))
    out["latency_ms"] = to_num(first_existing(raw, ["end_to_end_latency_ms", "latency_ms"]))
    out["tokens_per_sec"] = to_num(first_existing(raw, ["tokens_per_sec"]))
    out["cpu_percent_avg"] = to_num(first_existing(raw, ["cpu_percent_avg"]))
    out["cpu_percent_peak"] = to_num(first_existing(raw, ["cpu_percent_peak"]))
    out["ram_used_mb_peak"] = to_num(first_existing(raw, ["ram_used_mb_peak"]))
    out["process_ram_mb_peak"] = to_num(first_existing(raw, ["process_ram_mb_peak"]))
    out["temperature_c_avg"] = to_num(first_existing(raw, ["temp_c_avg", "temperature_c_avg"]))
    out["temperature_c_peak"] = to_num(first_existing(raw, ["temp_c_peak", "temperature_c_peak"]))
    out["accuracy_per_second"] = safe_div(out["accuracy"], out["latency_ms"] / 1000.0)
    return out


In [5]:

def expand_system_traces(samples_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if samples_df.empty or "system_metrics_trace_json" not in samples_df.columns:
        return pd.DataFrame()
    meta_cols = ["platform", "dataset", "model", "model_label", "run_id", "sample_idx"]
    for _, row in samples_df.iterrows():
        raw = row.get("system_metrics_trace_json")
        if pd.isna(raw) or raw in ("", None):
            continue
        try:
            points = json.loads(raw)
        except Exception:
            continue
        if not isinstance(points, list):
            continue
        for trace_idx, point in enumerate(points):
            if not isinstance(point, dict):
                continue
            rec = {c: row.get(c) for c in meta_cols if c in samples_df.columns}
            rec.update({
                "trace_idx": trace_idx,
                "t_s": point.get("t_s", np.nan),
                "phase": point.get("phase", np.nan),
                "cpu_percent": point.get("cpu_percent"),
                "ram_used_mb": point.get("ram_used_mb"),
                "ram_percent": point.get("ram_percent"),
                "process_ram_mb": point.get("process_ram_mb"),
                "temperature_c": point.get("temperature_c"),
            })
            rows.append(rec)
    traces = pd.DataFrame(rows)
    for col in ["trace_idx", "t_s", "cpu_percent", "ram_used_mb", "ram_percent", "process_ram_mb", "temperature_c"]:
        if col in traces.columns:
            traces[col] = to_num(traces[col])
    return traces


def summarize_for_visualization(summary: pd.DataFrame, samples: pd.DataFrame, traces: pd.DataFrame):
    group_cols = ["platform", "dataset", "model", "model_label"]
    sample_aggs = {
        "sample_count": ("sample_idx", "count"),
        "accuracy": ("accuracy", "mean"),
        "latency_ms_mean": ("latency_ms", "mean"),
        "latency_ms_median": ("latency_ms", "median"),
        "latency_ms_p95": ("latency_ms", lambda x: x.quantile(0.95)),
        "ttft_ms_mean": ("ttft_ms", "mean"),
        "tokens_per_sec_mean": ("tokens_per_sec", "mean"),
        "tokens_per_sec_median": ("tokens_per_sec", "median"),
        "output_tokens_mean": ("output_tokens", "mean"),
        "cpu_percent_avg_mean": ("cpu_percent_avg", "mean"),
        "cpu_percent_peak_max": ("cpu_percent_peak", "max"),
        "ram_used_mb_peak_max": ("ram_used_mb_peak", "max"),
        "process_ram_mb_peak_max": ("process_ram_mb_peak", "max"),
        "temperature_c_avg_mean": ("temperature_c_avg", "mean"),
        "temperature_c_peak_max": ("temperature_c_peak", "max"),
        "accuracy_per_second": ("accuracy_per_second", "mean"),
    }
    usable_aggs = {k: v for k, v in sample_aggs.items() if v[0] in samples.columns}
    model_dataset_summary = samples.groupby(group_cols, dropna=False).agg(**usable_aggs).reset_index() if not samples.empty and usable_aggs else pd.DataFrame()

    if not traces.empty:
        trace_summary = traces.groupby(group_cols, dropna=False).agg(
            trace_points=("trace_idx", "count"),
            trace_cpu_percent_mean=("cpu_percent", "mean"),
            trace_cpu_percent_peak=("cpu_percent", "max"),
            trace_ram_used_mb_mean=("ram_used_mb", "mean"),
            trace_ram_used_mb_peak=("ram_used_mb", "max"),
            trace_process_ram_mb_mean=("process_ram_mb", "mean"),
            trace_process_ram_mb_peak=("process_ram_mb", "max"),
            trace_temperature_c_mean=("temperature_c", "mean"),
            trace_temperature_c_peak=("temperature_c", "max"),
        ).reset_index()
    else:
        trace_summary = pd.DataFrame()

    ranking_cols = ["platform", "dataset", "model", "model_label", "accuracy", "latency_ms_mean", "tokens_per_sec_mean", "ram_used_mb_peak_max", "temperature_c_peak_max", "accuracy_per_second"]
    ranking = model_dataset_summary[[c for c in ranking_cols if c in model_dataset_summary.columns]].copy() if not model_dataset_summary.empty else pd.DataFrame()
    if not ranking.empty:
        ranking = ranking.sort_values(["dataset", "accuracy", "tokens_per_sec_mean", "latency_ms_mean"], ascending=[True, False, False, True])
    return model_dataset_summary, trace_summary, ranking


summary = normalize_summary(raw_summary)
samples = normalize_samples(raw_samples)
system_traces = expand_system_traces(samples)
model_dataset_summary, system_trace_summary, ranking_by_dataset = summarize_for_visualization(summary, samples, system_traces)

print("summary", summary.shape)
print("samples", samples.shape)
print("system_traces", system_traces.shape)
print("model_dataset_summary", model_dataset_summary.shape)
display(summary.head())
display(samples.head())
display(model_dataset_summary.head())


summary (24, 31)
samples (17800, 33)
system_traces (978140, 14)
model_dataset_summary (16, 20)


,platform,dataset,run_id,run_dir,model,model_label,model_dir,source_file,accuracy,accuracy_sd,ttft_ms,ttft_ms_sd,latency_ms,latency_ms_sd,tokens_per_sec,tokens_per_sec_sd,sustained_tokens_per_sec,sustained_tokens_per_sec_sd,samples_per_sec,samples_per_sec_sd,cpu_percent_avg,cpu_percent_avg_sd,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temperature_c_avg,temperature_c_avg_sd,temperature_c_peak,has_accuracy,accuracy_per_second,accuracy_per_1k_ms
0,pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,0.564,0.0117,5798.4835,23.7815,6552.3542,31.5635,3.5501,0.0369,0.3417,0.0037,0.1325,0.0006,72.3463,0.1756,100.0,6517.08,7265.75,67.0221,0.5420,71.9,True,0.086076,0.086076
1,pi5,hellaswag,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,0.489,0.0088,8085.0183,26.8886,8861.6988,28.6326,3.6275,0.0305,0.2997,0.0034,0.1015,0.0003,72.6253,0.3691,100.0,6939.64,7507.83,67.2594,0.1135,71.4,True,0.055181,0.055181
2,pi5,dolly,2026-05-01_09-41-37,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN
3,pi5,gsm8k,2026-05-01_15-01-51,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,0.640,0.0000,2879.2924,0.0000,42777.9699,0.0000,3.0606,0.0000,2.7010,0.0000,0.0223,0.0000,98.5461,0.0000,100.0,7010.58,7236.72,66.6654,0.0000,69.7,True,0.014961,0.014961
4,pi5,dolly,2026-06-27_18-19-44,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN


,platform,dataset,run_id,run_dir,model,model_label,model_dir,source_file,served_model_id,model_path,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,raw_output,system_metrics_trace_json,correct,accuracy,output_tokens,ttft_ms,latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temperature_c_avg,temperature_c_peak,accuracy_per_second
0,pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,1,A disk of radius 0.1 m initially at rest under...,A disk of radius 0.1 m initially at rest under...,0.4 m || 0.8 m || 1.2 m || 1.6 m,NaN,NaN,3.0,C,C. 1.,"[{""cpu_percent"":0.7,""ram_used_mb"":5054.59,""ram...",False,0.0,2,4587.0487,5453.5665,2.3081,78.2286,100.0,5054.59,6070.88,62.1000,63.7,0.000000
1,pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,2,"Cultural homogenization.\n\nA. 1,3,4\nB. 1,2,3...",Cultural homogenization.,"1,3,4 || 1,2,3 || 1,2,3,4 || 2,3,4",NaN,NaN,1.0,B,"B. 1,","[{""cpu_percent"":1.7,""ram_used_mb"":5049.56,""ram...",True,1.0,2,2882.0211,3741.2235,2.3277,71.0143,100.0,5066.02,6078.38,62.3571,63.7,0.267292
2,pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,3,Monte Alban is located in the ________ and was...,Monte Alban is located in the ________ and was...,Valley of Mexico; Aztec. || Valley of Oaxaca; ...,NaN,NaN,1.0,A,A. Valley of Mexico,"[{""cpu_percent"":0.5,""ram_used_mb"":5060.81,""ram...",False,0.0,4,3405.3366,4271.1300,4.6200,71.0429,100.0,5105.27,6098.95,62.3429,63.7,0.000000
3,pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,4,During a clinical study examining the effects ...,During a clinical study examining the effects ...,Pulse pressure || Stroke volume || Systolic bl...,NaN,NaN,3.0,C,C. Systolic,"[{""cpu_percent"":0.0,""ram_used_mb"":5095.39,""ram...",False,0.0,2,5416.0967,6283.6773,2.3053,70.7857,100.0,5095.39,6107.67,63.0429,64.8,0.000000
4,pi5,mmlu,2026-04-30_23-43-28,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M,/home/stephane/llm-experiments/results/pi5/Phi...,Phi-4-mini-instruct-Q4_K_M.gguf,models/Phi-4-mini-instruct-Q4_K_M.gguf,5,A 17-year-old girl is brought to the physician...,A 17-year-old girl is brought to the physician...,Hyperthyroidism || Hypogonadotropic hypogonadi...,NaN,NaN,1.0,C,C. Hypothyroidism,"[{""cpu_percent"":0.2,""ram_used_mb"":5092.81,""ram...",False,0.0,2,8947.6761,9823.7464,2.2829,70.9857,100.0,5102.88,6107.67,63.6143,65.9,0.000000


,platform,dataset,model,model_label,sample_count,accuracy,latency_ms_mean,latency_ms_median,latency_ms_p95,ttft_ms_mean,tokens_per_sec_mean,tokens_per_sec_median,output_tokens_mean,cpu_percent_avg_mean,cpu_percent_peak_max,ram_used_mb_peak_max,process_ram_mb_peak_max,temperature_c_avg_mean,temperature_c_peak_max,accuracy_per_second
0,pi5,dolly,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,1000,NaN,39450.126231,39180.80025,66073.249285,5181.743649,3.317602,3.4269,115.796,97.096649,100.0,7674.98,7565.58,68.926500,75.2,NaN
1,pi5,dolly,SmolLM3-3B-Q4_K_M.gguf,pi5 | SmolLM3-3B-Q4_K_M.gguf,1000,NaN,37000.970049,44524.36630,55687.093245,4461.405150,4.138388,4.2819,137.230,97.280343,100.0,5934.84,7380.84,67.777756,74.7,NaN
2,pi5,dolly,gemma-3-1b-it-Q8_0.gguf,pi5 | gemma-3-1b-it-Q8_0.gguf,1000,NaN,17764.864843,24679.00315,26318.962555,1208.778756,6.583540,6.7263,111.539,94.557700,100.0,2782.97,3477.75,65.357041,72.5,NaN
3,pi5,dolly,llama-3.2-1b-instruct-q4_k_m.gguf,pi5 | llama-3.2-1b-instruct-q4_k_m.gguf,1000,NaN,12406.993811,11956.75735,21765.263095,1512.943102,9.383390,9.5795,103.285,93.008159,100.0,2551.44,2994.56,66.824318,73.6,NaN
4,pi5,gsm8k,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,1000,0.638,41449.026976,42014.81880,62454.210730,2868.571539,3.142584,3.2002,119.844,98.447715,100.0,7203.09,7464.84,68.719736,72.5,0.020171


## Metric catalog with units

In [6]:

METRIC_CATALOG = pd.DataFrame([
    {"metric": "accuracy", "unit": "fraction", "level": "sample/summary", "meaning": "Correctness rate; only MMLU/HellaSwag/GSM8K"},
    {"metric": "latency_ms", "unit": "ms", "level": "sample/summary", "meaning": "End-to-end latency per request"},
    {"metric": "ttft_ms", "unit": "ms", "level": "sample/summary", "meaning": "Time to first token"},
    {"metric": "tokens_per_sec", "unit": "tokens/s", "level": "sample/summary", "meaning": "Decode throughput after first token"},
    {"metric": "sustained_tokens_per_sec", "unit": "tokens/s", "level": "summary", "meaning": "Sequential sustained throughput over full measured dataset run"},
    {"metric": "samples_per_sec", "unit": "samples/s", "level": "summary", "meaning": "Sequential sustained sample throughput"},
    {"metric": "output_tokens", "unit": "tokens", "level": "sample", "meaning": "Whitespace-estimated generated tokens"},
    {"metric": "cpu_percent_avg", "unit": "%", "level": "sample/summary", "meaning": "Average CPU utilization sampled during generation"},
    {"metric": "cpu_percent_peak", "unit": "%", "level": "sample/summary", "meaning": "Peak CPU utilization sampled during generation"},
    {"metric": "ram_used_mb_peak", "unit": "MB", "level": "sample/summary", "meaning": "Peak system RAM used"},
    {"metric": "process_ram_mb_peak", "unit": "MB", "level": "sample/summary", "meaning": "Peak llama-server process RSS"},
    {"metric": "temperature_c_avg", "unit": "°C", "level": "sample/summary", "meaning": "Average temperature sampled during generation"},
    {"metric": "temperature_c_peak", "unit": "°C", "level": "sample/summary", "meaning": "Peak temperature sampled during generation"},
    {"metric": "accuracy_per_second", "unit": "accuracy/s", "level": "derived", "meaning": "Accuracy divided by latency seconds; only scored datasets"},
])

display(METRIC_CATALOG)


,metric,unit,level,meaning
0,accuracy,fraction,sample/summary,Correctness rate; only MMLU/HellaSwag/GSM8K
1,latency_ms,ms,sample/summary,End-to-end latency per request
2,ttft_ms,ms,sample/summary,Time to first token
3,tokens_per_sec,tokens/s,sample/summary,Decode throughput after first token
4,sustained_tokens_per_sec,tokens/s,summary,Sequential sustained throughput over full meas...
5,samples_per_sec,samples/s,summary,Sequential sustained sample throughput
6,output_tokens,tokens,sample,Whitespace-estimated generated tokens
7,cpu_percent_avg,%,sample/summary,Average CPU utilization sampled during generation
8,cpu_percent_peak,%,sample/summary,Peak CPU utilization sampled during generation
9,ram_used_mb_peak,MB,sample/summary,Peak system RAM used


## Quick validation views

In [7]:

print("Available datasets in samples:", sorted(samples["dataset"].dropna().unique()) if "dataset" in samples.columns and not samples.empty else [])
print("Scored datasets:", SCORED_DATASETS)

if not ranking_by_dataset.empty:
    display(ranking_by_dataset.head(30))

for dataset in DATASETS:
    if not model_dataset_summary.empty and dataset in set(model_dataset_summary["dataset"]):
        print(f"\n{dataset.upper()} model/dataset summary")
        display(model_dataset_summary[model_dataset_summary["dataset"] == dataset].head(20))


Available datasets in samples: ['dolly', 'gsm8k', 'hellaswag', 'mmlu']
Scored datasets: ['mmlu', 'hellaswag', 'gsm8k']


,platform,dataset,model,model_label,accuracy,latency_ms_mean,tokens_per_sec_mean,ram_used_mb_peak_max,temperature_c_peak_max,accuracy_per_second
3,pi5,dolly,llama-3.2-1b-instruct-q4_k_m.gguf,pi5 | llama-3.2-1b-instruct-q4_k_m.gguf,NaN,12406.993811,9.383390,2551.44,73.6,NaN
2,pi5,dolly,gemma-3-1b-it-Q8_0.gguf,pi5 | gemma-3-1b-it-Q8_0.gguf,NaN,17764.864843,6.583540,2782.97,72.5,NaN
1,pi5,dolly,SmolLM3-3B-Q4_K_M.gguf,pi5 | SmolLM3-3B-Q4_K_M.gguf,NaN,37000.970049,4.138388,5934.84,74.7,NaN
0,pi5,dolly,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,NaN,39450.126231,3.317602,7674.98,75.2,NaN
4,pi5,gsm8k,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,0.638000,41449.026976,3.142584,7203.09,72.5,0.020171
5,pi5,gsm8k,SmolLM3-3B-Q4_K_M.gguf,pi5 | SmolLM3-3B-Q4_K_M.gguf,0.591000,40657.760078,3.947731,5674.92,71.9,0.015972
7,pi5,gsm8k,llama-3.2-1b-instruct-q4_k_m.gguf,pi5 | llama-3.2-1b-instruct-q4_k_m.gguf,0.519474,15998.565346,9.502284,2507.66,70.3,0.040376
6,pi5,gsm8k,gemma-3-1b-it-Q8_0.gguf,pi5 | gemma-3-1b-it-Q8_0.gguf,0.477368,23506.417344,6.118005,2632.81,69.2,0.022842
8,pi5,hellaswag,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,0.489000,8861.698752,3.627523,6939.64,71.4,0.060323
10,pi5,hellaswag,gemma-3-1b-it-Q8_0.gguf,pi5 | gemma-3-1b-it-Q8_0.gguf,0.291000,2055.944102,5.275044,2765.84,67.5,0.155579



MMLU model/dataset summary


,platform,dataset,model,model_label,sample_count,accuracy,latency_ms_mean,latency_ms_median,latency_ms_p95,ttft_ms_mean,tokens_per_sec_mean,tokens_per_sec_median,output_tokens_mean,cpu_percent_avg_mean,cpu_percent_peak_max,ram_used_mb_peak_max,process_ram_mb_peak_max,temperature_c_avg_mean,temperature_c_peak_max,accuracy_per_second
12,pi5,mmlu,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,1000,0.564,6552.354238,4841.08225,13456.387775,5798.483470,3.550143,3.48990,2.578,72.346263,100.0,6517.08,7265.75,67.022081,71.9,0.115925
13,pi5,mmlu,SmolLM3-3B-Q4_K_M.gguf,pi5 | SmolLM3-3B-Q4_K_M.gguf,1000,0.520,5666.356554,4212.78300,11334.254415,5051.321125,4.641600,4.77015,2.773,64.028524,100.0,3987.86,5381.48,66.170106,71.4,0.119948
14,pi5,mmlu,gemma-3-1b-it-Q8_0.gguf,pi5 | gemma-3-1b-it-Q8_0.gguf,1000,0.315,1566.485522,1233.97990,3039.883380,1378.339597,5.405407,5.20200,1.000,65.364234,100.0,2548.55,3018.64,59.919533,67.5,0.261954
15,pi5,mmlu,llama-3.2-1b-instruct-q4_k_m.gguf,pi5 | llama-3.2-1b-instruct-q4_k_m.gguf,1000,0.395,1775.790762,1270.86550,3747.923885,1692.702598,12.697066,13.61125,1.000,65.807132,100.0,1968.19,2294.22,64.243467,70.8,0.299505



HELLASWAG model/dataset summary


,platform,dataset,model,model_label,sample_count,accuracy,latency_ms_mean,latency_ms_median,latency_ms_p95,ttft_ms_mean,tokens_per_sec_mean,tokens_per_sec_median,output_tokens_mean,cpu_percent_avg_mean,cpu_percent_peak_max,ram_used_mb_peak_max,process_ram_mb_peak_max,temperature_c_avg_mean,temperature_c_peak_max,accuracy_per_second
8,pi5,hellaswag,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,1000,0.489,8861.698752,9871.05520,12821.608340,8085.018331,3.627523,4.46895,2.953,72.625256,100.0,6939.64,7507.83,67.259434,71.4,0.060323
9,pi5,hellaswag,SmolLM3-3B-Q4_K_M.gguf,pi5 | SmolLM3-3B-Q4_K_M.gguf,1000,0.290,7445.286190,8463.74375,10674.875240,6813.523776,4.877646,5.74245,3.080,62.374879,100.0,4530.36,5925.59,66.882700,71.4,0.051692
10,pi5,hellaswag,gemma-3-1b-it-Q8_0.gguf,pi5 | gemma-3-1b-it-Q8_0.gguf,1000,0.291,2055.944102,2245.77755,2962.920275,1864.609096,5.275044,5.17225,1.000,65.265334,100.0,2765.84,3369.88,62.352198,67.5,0.155579
11,pi5,hellaswag,llama-3.2-1b-instruct-q4_k_m.gguf,pi5 | llama-3.2-1b-instruct-q4_k_m.gguf,1000,0.251,2432.445084,2741.71340,3597.199240,2347.261102,12.359258,13.17040,1.000,65.667168,100.0,2161.02,2498.47,65.281101,70.3,0.146570



GSM8K model/dataset summary


,platform,dataset,model,model_label,sample_count,accuracy,latency_ms_mean,latency_ms_median,latency_ms_p95,ttft_ms_mean,tokens_per_sec_mean,tokens_per_sec_median,output_tokens_mean,cpu_percent_avg_mean,cpu_percent_peak_max,ram_used_mb_peak_max,process_ram_mb_peak_max,temperature_c_avg_mean,temperature_c_peak_max,accuracy_per_second
4,pi5,gsm8k,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,1000,0.638000,41449.026976,42014.81880,62454.210730,2868.571539,3.142584,3.20020,119.844000,98.447715,100.0,7203.09,7464.84,68.719736,72.5,0.020171
5,pi5,gsm8k,SmolLM3-3B-Q4_K_M.gguf,pi5 | SmolLM3-3B-Q4_K_M.gguf,1000,0.591000,40657.760078,42613.78620,50805.523675,2621.235341,3.947731,3.97250,149.724000,98.565149,100.0,5674.92,7036.30,67.379985,71.9,0.015972
6,pi5,gsm8k,gemma-3-1b-it-Q8_0.gguf,pi5 | gemma-3-1b-it-Q8_0.gguf,1900,0.477368,23506.417344,25425.22085,26663.532160,756.389540,6.118005,6.16015,139.423158,97.222522,100.0,2632.81,3233.36,65.231151,69.2,0.022842
7,pi5,gsm8k,llama-3.2-1b-instruct-q4_k_m.gguf,pi5 | llama-3.2-1b-instruct-q4_k_m.gguf,1900,0.519474,15998.565346,16064.58480,21209.091615,849.561775,9.502284,9.52150,144.146316,96.462137,100.0,2507.66,2857.98,66.761885,70.3,0.040376



DOLLY model/dataset summary


,platform,dataset,model,model_label,sample_count,accuracy,latency_ms_mean,latency_ms_median,latency_ms_p95,ttft_ms_mean,tokens_per_sec_mean,tokens_per_sec_median,output_tokens_mean,cpu_percent_avg_mean,cpu_percent_peak_max,ram_used_mb_peak_max,process_ram_mb_peak_max,temperature_c_avg_mean,temperature_c_peak_max,accuracy_per_second
0,pi5,dolly,Phi-4-mini-instruct-Q4_K_M.gguf,pi5 | Phi-4-mini-instruct-Q4_K_M.gguf,1000,NaN,39450.126231,39180.80025,66073.249285,5181.743649,3.317602,3.4269,115.796,97.096649,100.0,7674.98,7565.58,68.926500,75.2,NaN
1,pi5,dolly,SmolLM3-3B-Q4_K_M.gguf,pi5 | SmolLM3-3B-Q4_K_M.gguf,1000,NaN,37000.970049,44524.36630,55687.093245,4461.405150,4.138388,4.2819,137.230,97.280343,100.0,5934.84,7380.84,67.777756,74.7,NaN
2,pi5,dolly,gemma-3-1b-it-Q8_0.gguf,pi5 | gemma-3-1b-it-Q8_0.gguf,1000,NaN,17764.864843,24679.00315,26318.962555,1208.778756,6.583540,6.7263,111.539,94.557700,100.0,2782.97,3477.75,65.357041,72.5,NaN
3,pi5,dolly,llama-3.2-1b-instruct-q4_k_m.gguf,pi5 | llama-3.2-1b-instruct-q4_k_m.gguf,1000,NaN,12406.993811,11956.75735,21765.263095,1512.943102,9.383390,9.5795,103.285,93.008159,100.0,2551.44,2994.56,66.824318,73.6,NaN


## Export visualization-ready tables

In [8]:

EXPORTS = {
    "runs.csv": runs_df,
    "raw_summary.csv": raw_summary,
    "raw_samples.csv": raw_samples,
    "summary.csv": summary,
    "samples.csv": samples,
    "system_traces.csv": system_traces,
    "system_trace_summary.csv": system_trace_summary,
    "model_dataset_summary.csv": model_dataset_summary,
    "ranking_by_dataset.csv": ranking_by_dataset,
    "metric_catalog.csv": METRIC_CATALOG,
}

for filename, df in EXPORTS.items():
    path = PROCESSED_DIR / filename
    df.to_csv(path, index=False)
    print("wrote", path)

try:
    for filename, df in EXPORTS.items():
        df.to_parquet(PROCESSED_DIR / filename.replace(".csv", ".parquet"), index=False)
    print("Parquet export complete")
except Exception as exc:
    print("Skipped parquet export:", exc)


wrote /home/stephane/llm-experiments/results/pi5/processed/pi5/runs.csv
wrote /home/stephane/llm-experiments/results/pi5/processed/pi5/raw_summary.csv
wrote /home/stephane/llm-experiments/results/pi5/processed/pi5/raw_samples.csv
wrote /home/stephane/llm-experiments/results/pi5/processed/pi5/summary.csv
wrote /home/stephane/llm-experiments/results/pi5/processed/pi5/samples.csv
wrote /home/stephane/llm-experiments/results/pi5/processed/pi5/system_traces.csv
wrote /home/stephane/llm-experiments/results/pi5/processed/pi5/system_trace_summary.csv
wrote /home/stephane/llm-experiments/results/pi5/processed/pi5/model_dataset_summary.csv
wrote /home/stephane/llm-experiments/results/pi5/processed/pi5/ranking_by_dataset.csv
wrote /home/stephane/llm-experiments/results/pi5/processed/pi5/metric_catalog.csv
Skipped parquet export: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the 

## VS Code export helper

Run the next cell to export this notebook to HTML from VS Code/Jupyter.

In [9]:
import subprocess, sys
notebook_name = Path(globals().get("__vsc_ipynb_file__", "01_results_analysis_pi5_visualization_ready.ipynb"))
try:
    subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "html", str(notebook_name)], check=True)
except Exception as exc:
    print("HTML export skipped. In VS Code, use: Jupyter: Export to HTML")
    print(exc)

HTML export skipped. In VS Code, use: Jupyter: Export to HTML
Command '['/home/stephane/llm-experiments/.venv/bin/python', '-m', 'jupyter', 'nbconvert', '--to', 'html', '/home/stephane/llm-experiments/notebooks/01_results_analysis_pi5_visualization_ready.ipynb']' returned non-zero exit status 1.


usage: jupyter.py [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
                  [--paths] [--json] [--debug]
                  [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: kernel kernelspec migrate run troubleshoot

Jupyter command `jupyter-nbconvert` not found.
